# 看懂 Agent 每一步怎麼跑

這份教材示範如何追蹤工作流程的開始、結束、下一步，以及每一步留下的資料。

In [ ]:
from pathlib import Path
import os, sys, subprocess

if not Path('agentic_sdk').exists():
    if not Path('Agentic-SDK').exists():
        subprocess.run(['git', 'clone', 'https://github.com/R300-AI/Agentic-SDK.git'], check=True)
    os.chdir('Agentic-SDK')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

## 建立一個會留下執行紀錄的流程

`event_callback` 會在每個節點開始與結束時被呼叫。你可以把它想成 notebook 版的 Runner 執行紀錄。

In [ ]:
from agentic_sdk import Workflow
from agentic_sdk.modules import DirectAnswerAction, KeywordRetrieve, PassThroughPerceive

workflow = Workflow(
    workflow_name='可追蹤問答 Agent',
    perceive=PassThroughPerceive(),
    retrieve=KeywordRetrieve(items=[
        {'keywords': ['sdk'], 'content': 'Agentic SDK 的流程可以被逐步觀察。'},
    ]),
    action=DirectAnswerAction(),
)

events = []
def collect_event(event):
    events.append({
        'phase': event.get('phase'),
        'module': event.get('module'),
        'next_module': event.get('next_module'),
        'visit_count': event.get('visit_count'),
    })

result = workflow.run('請介紹 SDK 的追蹤方式', event_callback=collect_event)
print(result.final_message)

In [ ]:
for index, event in enumerate(events, start=1):
    print(index, event)

print('跑過哪些節點:', result.visit_counts)
print('中間資料:', result.entities)

## 下一步

如果 Agent 答得不如預期，先看它走過哪些節點、查到了什麼、最後回覆前留下哪些中間資料。